# Image Restoration: Visualize Results

This notebook is for qualitative inspection of the trained restoration model. It visualizes degraded inputs, restored outputs, and—when matching ground truth is available - PSNR, SSIM, and error maps.

**This notebook does not train the model.** `train.py` reproduces training, while `evaluate.py` is the standalone benchmark/inference entry point required by the hackathon.

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

PROJECT_ROOT = Path('..').resolve()
SAMPLE_INPUT_DIR = PROJECT_ROOT / 'sample_inputs'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

GT_DIR = None

NUM_SAMPLES = 6
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

print('Project root:', PROJECT_ROOT)
print('Input dir   :', SAMPLE_INPUT_DIR)
print('Output dir  :', OUTPUT_DIR)
print('GT dir      :', GT_DIR)

In [ ]:
def load_npy(path):
    arr = np.load(path).astype(np.float32)
    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f'Expected a 2D array after squeeze, got {arr.shape} for {path}')
    return arr

def display_image(arr):
    return np.clip(arr, 0.0, 1.0)

input_files = sorted(SAMPLE_INPUT_DIR.glob('*.npy'))
pairs = [(p, OUTPUT_DIR / p.name) for p in input_files if (OUTPUT_DIR / p.name).exists()]

if not pairs:
    raise FileNotFoundError(
        'No matching input/output .npy pairs found. Put test/sample inputs in sample_inputs/ '
        'and restored results in outputs/, or update the paths above.'
    )

print(f'Found {len(pairs)} input/output pairs.')

## 1. Input => Restored

This is the main qualitative result view: the degraded image is shown beside the model output.

In [ ]:
n = min(NUM_SAMPLES, len(pairs))
selected = rng.sample(pairs, n) if len(pairs) > n else pairs

fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
if n == 1:
    axes = np.expand_dims(axes, 0)

for r, (input_path, output_path) in enumerate(selected):
    inp = display_image(load_npy(input_path))
    out = display_image(load_npy(output_path))

    axes[r, 0].imshow(inp, cmap='gray', vmin=0, vmax=1)
    axes[r, 0].set_title(f'Input — {input_path.stem}')
    axes[r, 0].axis('off')

    axes[r, 1].imshow(out, cmap='gray', vmin=0, vmax=1)
    axes[r, 1].set_title(f'Restored — {output_path.stem}')
    axes[r, 1].axis('off')

plt.tight_layout()
plt.show()

## 2. Ground-truth evaluation (optional)

When `GT_DIR` points to matching ground-truth files, this section computes image-level PSNR and SSIM and displays error maps. This is useful on your validation set, not on a hidden/private test set where ground truth is unavailable.

In [ ]:
if GT_DIR is None:
    print('GT_DIR is None — skipping PSNR/SSIM and error-map analysis.')
else:
    gt_dir = Path(GT_DIR)
    rows = []
    gt_examples = []

    for input_path, output_path in pairs:
        gt_path = gt_dir / input_path.name
        if not gt_path.exists():
            continue

        pred = display_image(load_npy(output_path))
        gt = display_image(load_npy(gt_path))
        inp = display_image(load_npy(input_path))

        if pred.shape != gt.shape:
            raise ValueError(f'Shape mismatch for {input_path.name}: pred={pred.shape}, gt={gt.shape}')

        rows.append({
            'name': input_path.stem,
            'PSNR_dB': peak_signal_noise_ratio(gt, pred, data_range=1.0),
            'SSIM': structural_similarity(gt, pred, data_range=1.0),
        })
        if len(gt_examples) < 3:
            gt_examples.append((inp, pred, gt, input_path.stem))

    if not rows:
        print('No matching ground-truth files found.')
    else:
        import pandas as pd
        df = pd.DataFrame(rows).sort_values('PSNR_dB', ascending=False)
        display(df.style.format({'PSNR_dB':'{:.3f}', 'SSIM':'{:.4f}'}))
        print(f"Mean PSNR: {df['PSNR_dB'].mean():.3f} dB")
        print(f"Mean SSIM: {df['SSIM'].mean():.4f}")

        fig, axes = plt.subplots(len(gt_examples), 4, figsize=(16, 4 * len(gt_examples)))
        if len(gt_examples) == 1:
            axes = np.expand_dims(axes, 0)

        for r, (inp, pred, gt, name) in enumerate(gt_examples):
            error = np.abs(pred - gt)
            axes[r,0].imshow(inp, cmap='gray', vmin=0, vmax=1)
            axes[r,0].set_title('Input')
            axes[r,0].axis('off')

            axes[r,1].imshow(pred, cmap='gray', vmin=0, vmax=1)
            axes[r,1].set_title('Restored')
            axes[r,1].axis('off')

            axes[r,2].imshow(gt, cmap='gray', vmin=0, vmax=1)
            axes[r,2].set_title('Ground Truth')
            axes[r,2].axis('off')

            axes[r,3].imshow(error, cmap='magma', vmin=0, vmax=max(float(error.max()), 1e-6))
            axes[r,3].set_title('Absolute Error')
            axes[r,3].axis('off')

        plt.tight_layout()
        plt.show()

## 3. Intensity-distribution sanity check

This checks whether restoration outputs stay in the expected `[0, 1]` range and gives a quick view of how the output intensity distribution differs from the degraded input.

In [ ]:
input_values = np.concatenate([load_npy(a).ravel() for a, _ in pairs])
output_values = np.concatenate([load_npy(b).ravel() for _, b in pairs])

plt.figure(figsize=(10, 5))
plt.hist(np.clip(input_values, 0, 1), bins=80, alpha=0.5, density=True, label='Input')
plt.hist(np.clip(output_values, 0, 1), bins=80, alpha=0.5, density=True, label='Restored')
plt.xlabel('Normalized intensity')
plt.ylabel('Density')
plt.title('Input vs Restored intensity distribution')
plt.legend()
plt.tight_layout()
plt.show()